# Inferencia sobre video

Etapa de inferencia del pipeline, hasta el CSV. La lógica de alerta va aparte (`alerta.py`) y se calcula después, así τ, K y N se barren sin volver a procesar el video.

1. **Video → detecciones.** Cada frame pasa por el detector con el **umbral de confianza bajo** del pipeline (`CONF_MIN = 0.05`, prioriza recall) y NMS de Ultralytics. El letterbox lo hace Ultralytics con el mismo `imgsz` del run.
2. **Clasificación de recortes** (cuando exista `clasificador.ipynb`). Cada caja se recorta con `recortes.recortar` y el clasificador agrega una columna `p_real`.
3. **Vista previa de la alerta** con una configuración por defecto. El barrido completo va en `barrido_alerta.ipynb`.

Salidas en `results/video/<detector>/`:

- `<clip>_detecciones.csv`: una fila por detección (`frame, t, clase, conf, x1, y1, x2, y2[, p_real]`).
- `<clip>_frames.csv`: una fila por frame procesado, tenga o no detecciones (`frame, t, n_det, ms_*`). Es lo que necesita `alerta.simular`: un frame sin detecciones también cuenta en la ventana K-de-N.
- `<clip>_meta.json`: con qué pesos y parámetros se generó.

In [ ]:
import json
import time
from datetime import date
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from tqdm.auto import tqdm
from ultralytics import YOLO

import alerta
import recortes

REPO_ROOT = Path.cwd()
VIDEO_DIR = REPO_ROOT / "videos"
CLIPS_CSV = VIDEO_DIR / "clips.csv"
FIG_DIR = REPO_ROOT / "figures"
for d in (VIDEO_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu")

## 1. Configuración

- `RUN_NAME`: el detector entrenado (`runs/<RUN_NAME>/weights/best.pt`). Si todavía no está, se usa `yolo11n.pt` de COCO **solo para probar el pipeline**: sus clases no son fuego ni humo y esos CSV no sirven para métricas (quedan en una carpeta aparte, `coco_placeholder`).
- `IMGSZ`: el mismo `imgsz` con que se entrenó el run.
- `VID_STRIDE`: procesar 1 de cada `VID_STRIDE` frames. Ojo: N en K-de-N cuenta **frames procesados**, así que con stride 2 una ventana de 5 cubre el doble de tiempo.

In [ ]:
RUN_NAME = "yolo11n_640_neg_s42"
IMGSZ = 640
CONF_MIN = 0.05   # umbral bajo del pipeline; τ se aplica después, en alerta.py
IOU_NMS = 0.7     # NMS (valor por defecto de Ultralytics)
VID_STRIDE = 1
OVERWRITE = False # True: reprocesar aunque ya existan los CSV con los mismos parámetros

best_pt = REPO_ROOT / "runs" / RUN_NAME / "weights" / "best.pt"
if best_pt.exists():
    DET_WEIGHTS, DET_TAG = best_pt, RUN_NAME
else:
    DET_WEIGHTS, DET_TAG = "yolo11n.pt", "coco_placeholder"
    print(f"[!] No está {best_pt.relative_to(REPO_ROOT)}: uso yolo11n.pt de COCO solo para probar el pipeline.")

OUT_DIR = REPO_ROOT / "results" / "video" / DET_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
detector = YOLO(DET_WEIGHTS)
print("Detector:", DET_WEIGHTS, "| clases:", detector.names if len(detector.names) <= 5 else f"{len(detector.names)} (COCO)")

## 2. Clips

`videos/clips.csv` define los clips del plan: fuego claro, humo lento, nocturno y **sin evento** (para medir falsas alarmas).

| Columna | Contenido |
|---|---|
| `clip` | nombre corto, se usa en los archivos de salida |
| `archivo` | ruta del video relativa a `videos/` |
| `tipo` | `fuego`, `humo`, `nocturno` o `sin_evento` |
| `t_evento_s` | segundo en que el evento **se vuelve visible**, anotado a mano. Vacío para `sin_evento` |
| `notas` | libre |

Si el archivo no existe, se crea una plantilla. Los videos no se versionan (`videos/*.mp4` en `.gitignore`); `clips.csv` sí.

In [ ]:
TIPOS = {"fuego", "humo", "nocturno", "sin_evento"}

if not CLIPS_CSV.exists():
    pd.DataFrame({
        "clip": ["fuego_claro", "humo_lento", "nocturno", "sin_evento"],
        "archivo": ["", "", "", ""],
        "tipo": ["fuego", "humo", "nocturno", "sin_evento"],
        "t_evento_s": [np.nan, np.nan, np.nan, np.nan],
        "notas": ["", "", "", ""],
    }).to_csv(CLIPS_CSV, index=False)
    print(f"Plantilla creada en {CLIPS_CSV.relative_to(REPO_ROOT)}: completar archivo y t_evento_s.")

clips = pd.read_csv(CLIPS_CSV, dtype={"archivo": str, "notas": str}).fillna({"archivo": "", "notas": ""})
assert clips["clip"].is_unique, "Nombres de clip repetidos"
assert set(clips["tipo"]) <= TIPOS, f"Tipos válidos: {TIPOS}"
con_evento = clips["tipo"] != "sin_evento"
assert clips.loc[~con_evento, "t_evento_s"].isna().all(), "Los clips sin_evento no llevan t_evento_s"

clips["ruta"] = [VIDEO_DIR / a if a else None for a in clips["archivo"]]
clips["disponible"] = [r is not None and r.exists() for r in clips["ruta"]]
falta_t = con_evento & clips["t_evento_s"].isna() & clips["disponible"]
if falta_t.any():
    print("[!] Falta anotar t_evento_s (sin eso no hay latencia):", clips.loc[falta_t, "clip"].tolist())
clips[["clip", "archivo", "tipo", "t_evento_s", "disponible"]]

### Ayuda para anotar `t_evento_s`

`t_evento_s` es el segundo en que una persona mirando la cámara **podría** notar el humo o el fuego (no cuando lo detecta el modelo, ni cuando ya es obvio). Criterio:

1. Ir a un momento en que el evento se ve claro y **retroceder** hasta el último frame en que todavía se distingue (mirando hacia adelante, sabiendo dónde va a aparecer, se anota antes de lo que lo vería un operador).
2. A resolución original, sin zoom.
3. Anotado por **dos personas por separado**; si difieren en más de 1–2 s se discute, y se usa el promedio. La diferencia se reporta como incertidumbre de la latencia.
4. El clip tiene que empezar **antes** del evento (idealmente 30 s o más); si no, la latencia no se puede medir.

`tira_frames` muestra frames equiespaciados entre dos tiempos, con su timestamp, para ir acotando.

In [ ]:
def tira_frames(clip: str, t_desde: float, t_hasta: float, n: int = 8, columnas: int = 4):
    ruta = clips.set_index("clip").loc[clip, "ruta"]
    cap = cv2.VideoCapture(str(ruta))
    fps = cap.get(cv2.CAP_PROP_FPS)
    tiempos = np.linspace(t_desde, t_hasta, n)
    filas = int(np.ceil(n / columnas))
    fig, axes = plt.subplots(filas, columnas, figsize=(4 * columnas, 2.6 * filas), squeeze=False)
    for ax, t in zip(axes.flat, tiempos):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(round(t * fps)))
        ok, img = cap.read()
        if ok:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"t = {t:.2f} s", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[n:]:
        ax.axis("off")
    cap.release()
    fig.suptitle(clip)
    fig.tight_layout()

# Ejemplo: tira_frames("humo_lento", 10, 20)

## 3. Video → detecciones

Las ms se miden por frame: `ms_total` incluye pre y postprocesamiento (lo que ve el sistema) y `ms_inference` es solo el forward del modelo. Son orientativas: la latencia del paper sale de `benchmark.ipynb`.

Los timestamps salen de `frame / fps`, válido para video de fps constante (el caso de las cámaras de vigilancia).

In [ ]:
def params_inferencia() -> dict:
    return {
        "pesos": str(DET_WEIGHTS), "imgsz": IMGSZ, "conf_min": CONF_MIN,
        "iou_nms": IOU_NMS, "vid_stride": VID_STRIDE,
    }

def rutas_salida(clip: str) -> dict[str, Path]:
    return {k: OUT_DIR / f"{clip}_{k}.{ext}" for k, ext in (("detecciones", "csv"), ("frames", "csv"), ("meta", "json"))}

def inferir_clip(ruta: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    cap = cv2.VideoCapture(str(ruta))
    if not cap.isOpened():
        raise IOError(f"No se pudo abrir {ruta}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ancho, alto = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if fps <= 0:
        raise ValueError(f"{ruta}: fps inválido ({fps})")

    det_rows, frame_rows = [], []
    for idx in tqdm(range(n_total), desc=ruta.name, leave=False):
        if idx % VID_STRIDE:
            if not cap.grab():  # avanza sin decodificar
                break
            continue
        ok, img = cap.read()
        if not ok:
            break
        t = idx / fps
        t0 = time.perf_counter()
        r = detector.predict(img, imgsz=IMGSZ, conf=CONF_MIN, iou=IOU_NMS, device=DEVICE, verbose=False)[0]
        ms_total = (time.perf_counter() - t0) * 1000

        frame_rows.append({"frame": idx, "t": t, "n_det": len(r.boxes), "ms_total": ms_total,
                           **{f"ms_{k}": v for k, v in r.speed.items()}})
        for (x1, y1, x2, y2), conf, cls in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()):
            det_rows.append({"frame": idx, "t": t, "clase": r.names[int(cls)], "conf": conf,
                             "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    cap.release()

    meta = {
        **params_inferencia(), "video": str(ruta), "fps": fps, "n_frames_video": n_total,
        "n_frames_procesados": len(frame_rows), "ancho": ancho, "alto": alto,
        "duracion_s": n_total / fps, "device": str(DEVICE), "ultralytics": ultralytics.__version__,
        "fecha": date.today().isoformat(),
    }
    cols = ["frame", "t", "clase", "conf", "x1", "y1", "x2", "y2"]
    return pd.DataFrame(det_rows, columns=cols), pd.DataFrame(frame_rows), meta

def ya_procesado(clip: str) -> bool:
    rutas = rutas_salida(clip)
    if OVERWRITE or not all(p.exists() for p in rutas.values()):
        return False
    meta = json.loads(rutas["meta"].read_text(encoding="utf-8"))
    return all(meta.get(k) == v for k, v in params_inferencia().items())

resumen = []
for c in clips[clips["disponible"]].itertuples():
    rutas = rutas_salida(c.clip)
    if ya_procesado(c.clip):
        print(f"{c.clip}: ya procesado con estos parámetros, se saltea.")
    else:
        det, frames, meta = inferir_clip(c.ruta)
        det.to_csv(rutas["detecciones"], index=False)
        frames.to_csv(rutas["frames"], index=False)
        rutas["meta"].write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    meta = json.loads(rutas["meta"].read_text(encoding="utf-8"))
    frames = pd.read_csv(rutas["frames"])
    det = pd.read_csv(rutas["detecciones"])
    resumen.append({
        "clip": c.clip, "tipo": c.tipo, "duracion_s": round(meta["duracion_s"], 1), "fps_video": round(meta["fps"], 1),
        "frames_procesados": len(frames), "detecciones": len(det),
        "ms_total_medio": round(frames["ms_total"].mean(), 1),
        "fps_sistema": round(1000 / frames["ms_total"].mean(), 1),
    })

if not resumen:
    print("No hay clips disponibles: completar videos/clips.csv y copiar los videos a videos/.")
pd.DataFrame(resumen)

## 4. Clasificación de recortes (`p_real`)

Segunda etapa del pipeline: cada caja candidata se recorta del frame (mismo margen que en el entrenamiento del clasificador, `recortes.MARGEN`) y el clasificador da la probabilidad de que sea fuego o humo real. Se guarda como columna `p_real` en el CSV de detecciones; `alerta.ConfigAlerta(umbral_clf=...)` la usa para confirmar o descartar.

Se clasifican las detecciones con `conf > CLF_CONF_MIN` (el τ más chico del barrido, 0,25): por debajo nunca cuentan para la alerta y clasificarlas solo cuesta tiempo (con `CONF_MIN = 0.05` hay muchas cajas por frame). Así el barrido puede combinar cualquier τ ≥ 0,25 con cualquier umbral del clasificador; si se barre un τ menor, `alerta.py` falla avisando que falta `p_real`. Si todavía no existe el clasificador, la sección se saltea.

In [ ]:
CLF_WEIGHTS = REPO_ROOT / "runs" / "clasificador" / "weights" / "best.pt"  # sale de clasificador.ipynb
CLF_IMGSZ = 224
CLF_CONF_MIN = 0.25  # τ mínimo del barrido

def agregar_p_real(ruta_video: Path, rutas: dict[str, Path], clf: YOLO) -> pd.DataFrame:
    det = pd.read_csv(rutas["detecciones"])
    idx_real = recortes.indice_clase_real(clf.names)
    p_real = np.full(len(det), np.nan)
    a_clasificar = det[det["conf"] > CLF_CONF_MIN]
    if len(a_clasificar):
        # frame → posiciones (en det) de las cajas a clasificar
        por_frame = {f: det.index.get_indexer(g.index) for f, g in a_clasificar.groupby("frame")}
        cap = cv2.VideoCapture(str(ruta_video))
        ultimo = max(por_frame)
        for idx in tqdm(range(ultimo + 1), desc=f"clf {ruta_video.name}", leave=False):
            if idx not in por_frame:
                cap.grab()
                continue
            ok, img = cap.read()
            if not ok:
                break
            pos = por_frame[idx]
            crops = [recortes.recortar(img, *det.iloc[i][["x1", "y1", "x2", "y2"]]) for i in pos]
            res = clf.predict(crops, imgsz=CLF_IMGSZ, device=DEVICE, verbose=False)
            p_real[pos] = [float(r.probs.data[idx_real]) for r in res]
        cap.release()
    det["p_real"] = p_real
    det.to_csv(rutas["detecciones"], index=False)
    return det

if CLF_WEIGHTS.exists():
    clasificador = YOLO(CLF_WEIGHTS)
    for c in clips[clips["disponible"]].itertuples():
        det = agregar_p_real(c.ruta, rutas_salida(c.clip), clasificador)
        print(f"{c.clip}: p_real en {det['p_real'].notna().sum()} detecciones")
else:
    print(f"Sin clasificador todavía ({CLF_WEIGHTS.relative_to(REPO_ROOT)}): se usa solo el detector.")

## 5. Vista previa de la alerta

Una configuración por defecto (τ = 0,40, 3 de 5) para ver que el CSV y `alerta.py` encajan, y una línea de tiempo por clip: confianza máxima por frame y clase, umbral τ, segundo anotado del evento y períodos en ALERTA. El barrido completo de τ / K / N con y sin clasificador va en `barrido_alerta.ipynb`.

In [ ]:
CFG_PREVIEW = alerta.ConfigAlerta(tau=0.40, k=3, n=5)
COLORES = {"fire": "tab:red", "smoke": "tab:gray"}

metricas = []
disponibles = clips[clips["disponible"]]
fig, axes = plt.subplots(max(len(disponibles), 1), 1, figsize=(12, 2.6 * max(len(disponibles), 1)), squeeze=False)
for ax, c in zip(axes[:, 0], disponibles.itertuples()):
    rutas = rutas_salida(c.clip)
    det, frames = pd.read_csv(rutas["detecciones"]), pd.read_csv(rutas["frames"])
    meta = json.loads(rutas["meta"].read_text(encoding="utf-8"))
    estados, eventos = alerta.simular(frames, det, CFG_PREVIEW)
    metricas.append({"clip": c.clip, "tipo": c.tipo, "eventos": len(eventos),
                     **alerta.metricas_clip(eventos, meta["duracion_s"], c.t_evento_s)})

    for clase in CFG_PREVIEW.clases:
        conf_max = det[det["clase"] == clase].groupby("frame")["conf"].max().reindex(frames["frame"], fill_value=0)
        ax.plot(frames["t"], conf_max.values, lw=0.8, color=COLORES.get(clase), label=clase)
    en_alerta = (estados["estado"] == alerta.ALERTA).to_numpy()
    ax.fill_between(estados["t"], 0, 1, where=en_alerta, color="tab:orange", alpha=0.2, step="post", label="ALERTA")
    ax.axhline(CFG_PREVIEW.tau, color="k", ls=":", lw=0.8)
    if pd.notna(c.t_evento_s):
        ax.axvline(c.t_evento_s, color="tab:blue", ls="--", lw=1, label="evento visible")
    ax.set_ylim(0, 1)
    ax.set_ylabel("conf máx")
    ax.set_title(f"{c.clip} ({c.tipo}) — {CFG_PREVIEW.nombre}", fontsize=10)
axes[-1, 0].set_xlabel("t (s)")
axes[0, 0].legend(loc="upper right", fontsize=8, ncol=4)
fig.tight_layout()
if len(disponibles):
    fig.savefig(FIG_DIR / f"lineas_tiempo_{DET_TAG}.png", dpi=150, bbox_inches="tight")

pd.DataFrame(metricas)